<img src="https://radar.community.uaf.edu/wp-content/uploads/sites/667/2021/03/HydroSARbanner.jpg" width="100%" />

<br>
<font size="6"> <b>FIER Daily Flood Forecasting Code</b><img style="padding: 7px" src="https://radar.community.uaf.edu/wp-content/uploads/sites/667/2021/03/UAFLogo_A_647.png" width="170" align="right"/></font>

<br>
<font size="4"> <b> Franz J Meyer, University of Alaska Fairbanks</b> <br>
</font>

This notebook uses polynomial functions and neural network models to generate daily flood inundation predictions using time series of Sentinel-1 RTC data and GEOGLoWs river runoff forecasts. 
    
The workflow utilizes information available in the fierpy <a href="https://github.com/SERVIR/fierpy">fierpy</a> GitHub repository.
<hr>


# 1. Load Python Libraries

This code cell loads the necessary python libraries needed to run this forecasting notebook.

In [ ]:
# load needed modules
import _pickle as cPickle
from pathlib import Path
import time
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import rasterio
import re
from datetime import datetime, date
import pandas as pd
import os, datetime
import rioxarray as rxr
import csv
from datetime import datetime
import copy
import geoglows

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras import models
from tensorflow.keras.layers import Normalization
fontSize_new = 14
plt.rcParams.update({'font.size':fontSize_new})

***
# 2. Load the Pre-Trained Model 

## 2.1 Select Folder That Contains the Pre-Trained Model Data

We leverage the outcome of a pretrained forecasting model that was created using notebook ```a4-FloodTraining.ipynb```. Point your notebook to the folder containing your model data.

In [ ]:
from ipyfilechooser import FileChooser
fc = FileChooser(Path.cwd())
display(fc)

## 2.2 Load Model Parameters

In [ ]:
forecasting_files = "ForecastingFiles_Created-20251014_NSE=0.001_forwardOnly_5Modes-REOF_5Modes-Fit_thrshld=None_myDim=discharge_MDretained03-01to10-30_despiked_tA-filt.pkl"

In [ ]:
# Load the variables
start_reload = time.time()


if 'reof_ds' in globals():
    del reof_ds, modes_poly_smoothed, degs_smoothed, polys_smoothed, models_nn_smoothed, modes_nn_smoothed, q_smoothed, q_tot_smoothed, type_file, da_tot, ind_flood, floodpercent, time_flood
    print('Deleting old data...')
else:
    print('Loading data...')


with open(Path(fc.selected,"Forecasting-Files","tA-filter_test",forecasting_files), "rb") as f: 
# with open(Path(fc.selected,"Forecasting-Files",forecasting_files), "rb") as f: 
    reof_ds = cPickle.load(f) # works
    print('reof_ds complete')
    modes_poly_smoothed = cPickle.load(f) # works
    print('modes_poly_smoothed complete')
    degs_smoothed = cPickle.load(f) # works
    print('degs_smoothed complete')
    polys_smoothed = cPickle.load(f) # works
    print('polys_smoothed complete')
    models_nn_smoothed = cPickle.load(f) # works
    print('models_nn_smoothed complete')
    modes_nn_smoothed = cPickle.load(f) # works
    print('modes_nn_smoothed complete')
    q_smoothed = cPickle.load(f) # works
    print('q_smoothed complete')
    q_tot_smoothed = cPickle.load(f) # works
    print('q_tot_smoothed complete')
    type_file = cPickle.load(f) # works
    print('type_file complete')
    da_tot = cPickle.load(f) # works
    print('da_tot complete')
    ind_flood = cPickle.load(f) # works
    print('ind_flood complete')
    floodpercent = cPickle.load(f) # works
    print('floodpercent complete')
    time_flood = cPickle.load(f) # works
    print('time_flood complete')
    lat = cPickle.load(f)
    lon = cPickle.load(f)
    print('lat & lon complete')
    ReachID = cPickle.load(f)
    print('ReachID complete')
    model_sel = cPickle.load(f)
    print('Model selection complete')
    CRS = cPickle.load(f)
    print('CRS complete')
    transform = cPickle.load(f)
    print('transform complete')
    smoothing_frame = cPickle.load(f)
    print('smoothing_frame loaded')
    scores_nn = cPickle.load(f)
    print('scores_nn loaded')
    scores_poly = cPickle.load(f)
    print('scores_poly loaded')
    NSEs_poly_smoothed = cPickle.load(f)
    print('NSEs_poly_smoothed loaded')
    NSEs_nn_smoothed = cPickle.load(f)
    print('NSEs_nn_smoothed loaded')
    myIdx = cPickle.load(f)
    print('myIdx loaded')


end_reload = time.time()

print('It took ',end_reload-start_reload,' seconds to load the variables.')

***
# 3. Define Processing Parameters

Next we define processing parameters such as 
* Potential forward aggregation of discharge data
* Buffering of the hydrobasin boundaries
* Where to store the results of the forecast.

## 3.1 Set Forward Aggregation Window Length

Set filter length using variable ```myIdx```. Choose ```myIdx = 0``` for no filtering.

In [ ]:
myIdx_original = copy.deepcopy(myIdx)
print(f'Best Index: {myIdx_original}')
print(f'Max Index: {len(smoothing_frame)-1}')

In [ ]:
import sys

myIdx = len(smoothing_frame)-1 # this will be changed from 0 up to the maximum number of smoothings
# myIdx = myIdx_original

if myIdx > len(smoothing_frame)-1:
    raise Exception(f'\nWARNING!!!\nWARNING!!!\nWARNING!!!\nmyIdx is greater than the highest smoothing frame index!\nmyIdx = {myIdx}, max frame index = {len(smoothing_frame)-1}')

if len(smoothing_frame) >= 2:
    q_tot = q_tot_smoothed[myIdx].copy(deep=True) 
else:
    q_tot = q_tot_smoothed.copy(deep=True)
    
# polynomial
modes_poly = copy.deepcopy(modes_poly_smoothed[myIdx])
degs = copy.deepcopy(degs_smoothed[myIdx])
polys = copy.deepcopy(polys_smoothed[myIdx])

# neural network
modes_nn = copy.deepcopy(modes_nn_smoothed[myIdx])
models_nn = copy.deepcopy(models_nn_smoothed[myIdx])

Set location for forecasting output.

In [ ]:
# generalized location in which to save files
myLoc = f'Forecast_forwardOnly-s_f={smoothing_frame[myIdx]:02d}days_{len(modes_nn)}modes-REOF_MDretained03-01to10-30_Created-{datetime.today().strftime("%Y%m%d")}'
myLoc = f'uncertaintyMap-test_{datetime.today().strftime("%Y%m%d")}'

debuff = False
if debuff:
    debuff_shapefile = FileChooser(Path.cwd())
    display(debuff_shapefile)

# set y axis limits for plots flood percentage (observed and forecasted) & discharge vs time; this allows for intercomparison between different runs. 
# This can be set to 'None' (which is the default) for y limits to automatically scale.
fp_axis_lims = [0, np.ceil(np.max(floodpercent)+5)]
# fp_axis_lims = None

In [ ]:
print(f'Your save location: {Path(fc.selected, "Figures", myLoc)}')
if debuff:
    print(f'\nYour debuff_shapefile location: {debuff_shapefile.selected}')

***
# 4. Define Important Functions

In [ ]:
### --- Z-Score calculation --- ###
def z_score(flood_forecast, flood_hindcast):
    # Calculate the indices for scoring: a = true positive, b = false positive, c = false negative, d = true negative
    a = np.sum(np.logical_and(flood_forecast == 1, flood_hindcast == 1))
    b = np.sum(np.logical_and(flood_forecast == 1, flood_hindcast == 0))
    c = np.sum(np.logical_and(flood_forecast == 0, flood_hindcast == 1))
    d = np.sum(np.logical_and(flood_forecast == 0, flood_hindcast == 0))

    # Calculate the skills
    overall_accuracy = ((a + d) / (a + b + c + d)) * 100
    CSI = (a / (a + b + c)) * 100 # Critical Success Index; fraction correct of observed
    precision = (a / (a + b)) * 100
    recall = (a / (a + c)) * 100 # Also known as Hit Rate (HR); tells you about over-prediction of wet areas
    FAR = (b / (b + d)) *100 # False Alarm Rate; tells you about over-prediction of dry areas. 
    PSS = recall - FAR # Pierce Skill Score; incorporates over-prediction of both dry and wet areas. 
    # PSS = 1; perfect forecast
    # PSS = 0; no skill, completely random.
    # PSS = -1; perfect inverse forecast. 
    
    # Replace NaNs by 0s
    if np.isnan(overall_accuracy):
        overall_accuracy = 0
    if np.isnan(CSI):
        CSI = 0
    if np.isnan(precision):
        precision = 0
    if np.isnan(recall):
        recall = 0
    if np.isnan(FAR):
        FAR = 0
    if np.isnan(PSS): 
        PSS = 0

    

    return overall_accuracy, CSI, precision, recall, FAR, PSS


### --- Forecast calculation --- ###

def partialForecastCalc(forecastType, myMode, myModes, q_forecast, reof_ds, models):
    # Calculate the water signal (without reof_ds.values.center) for a single mode. 
    # Returns a 3 dimensional matrix: forecast_noCenter[time, space (X), space (Y)]

    # Create the forecast based on a single spatial mode
    print(f'partialForecastCalc initiated.\nRunning for mode {myMode}...')

    # Get dimensions of datasets
    n_modes, y, x = reof_ds.spatial_modes.shape
    t = q_forecast.shape[0]

    # Calculate the temporal amplitudes based on the given models and input proxy forecast data (usually discharge)
    if 'p' in forecastType:
        forecast_rtcp = np.array([model(q_forecast.values) for model in models]).squeeze()
    elif 'n' in forecastType:
        forecast_rtcp = np.array([model.predict(q_forecast.values) for model in models]).squeeze() # original
    else:
        print('Input forecastType is neither neural network nor polynomial!')

    # Check the size of the forecasted rtcps. If it has only one dimension, add a new dimension
    if len(forecast_rtcp.shape) == 1:
        forecast_rtcp = forecast_rtcp[:, np.newaxis] # Add a new dimension
        forecast_rtcp = forecast_rtcp.T # Transpose to match the dimensions of the spatial modes    

    ### --- Construct the partial forecast by multiplying the spatial mode by the calculated temporal amplitude for the mode --- ###
    forecast_noCenter = np.dot(reof_ds.spatial_modes.values[myMode-1,:,:].reshape(-1,1), 
                               forecast_rtcp[myModes.index(myMode)].reshape(1,len(forecast_rtcp[myModes.index(myMode)]))).reshape(y,x,t).transpose(2,0,1)
    # Loop through each entry of forecast_noCenter and flip it; this is so it displays properly elsewhere. 
    for i in range(0,forecast_noCenter.shape[0]):
        forecast_noCenter[i,:,:] = np.flipud(forecast_noCenter[i,:,:])
        
    return forecast_noCenter


def finalForecastCalc(forecasts_noCenter, reof_ds, myModes):
    # Calculate the water signal (including reof_ds.values.center) for multiple modes. 
    # Returns an xarray containing the final forecast(s): da. 
    # forecasts_noCenter: 4 dimensional numpy array; [mode, time, space (Y dim), space (X dim)]
    # reof_ds: xarray containing the spatial modes and temporal amplitudes. 
    # myMode: list of integers representing the modes the user wishes to combine. 
    #         NOTE - the entries of 'myMode' must be subtracted by 1 to convert to python notation. 

    # Get dimensions of datasets
    n_modes, y, x = reof_ds.spatial_modes.shape
    t = q_forecast.shape[0]

    # Create the variable filled with zeros
    zeros_variable = np.zeros((t, y, x))
    
    # Create the new DataArray with zeros_variable
    da = xr.DataArray(
        zeros_variable,
        coords={'time': q_forecast.time, 'y': da_tot.y, 'x': da_tot.x},
        dims=['time', 'y', 'x']
    )

    # Combine relevant forecasts 
    if isinstance(myModes, int):
        myModes = [myModes]
    for myMode in myModes:
        print(f"myMode = {myMode}")
        da.values += forecasts_noCenter[myMode-1,:,:,:]
    da.values += reof_ds.center.values
    
    return da

### --- Visualization code --- ###


# define function to get flood percentages from each flood forecast. 
def floodPercentCalc(forecast): 
    # set up empty array into which to put the percentages
    floodPercent=np.array([])
    # set up the total number of forecasts and a counter to keep track of progress. 
    total = forecast.shape[0]
    county = 0
    print("Starting calcualtion")
    for cast in forecast:
        # get the unique elements and their counts
        unique_elements, counts = np.unique(cast, return_counts=True)
        myDict = dict(zip(unique_elements, counts))
        myArea = myDict[0] + myDict[1]
        flooded = myDict[1]
        floodPercent = np.append(floodPercent, flooded/myArea*100.0)
        county += 1
        print(f"{county} of {total} complete")
    return floodPercent


# define function to plot flood percentage and discharge through time
def plot_flood_and_discharge(forecasts, flood_percents, q_tot, floodpercent_real = None, time_real=None, saveIt=False, fp_lims = None):
    
    # set up figure and twin the axis
    fig_perc_dis, ax1 = plt.subplots(figsize=(9,6))
    ax2 = ax1.twinx()

    
    # plot the neural network flood percentages
    ln1 = ax1.plot(forecasts.time, flood_percents, color='blue', marker='o', label='Flood Percentage from Forecast')
    ax1.set_ylabel('Flood Percentage [%]')
    if fp_lims:
        ax1.set_ylim(fp_lims[0], fp_lims[1])
    ax1.grid()
    
    # plot the discharge values through same time. 
    idx1, idx2 = np.where(forecasts[0].time == q_tot.time)[0][0], np.where(forecasts[-1].time == q_tot.time)[0][0]
    ln2 = ax2.plot(q_tot[idx1:idx2+1].time, q_tot[idx1:idx2+1], color='red', marker='^', label='Discharge from GEOGLOWS')
    ax2.set_ylabel('Discharge [$m^3/s$]')

    lns = ln1 + ln2

    # plot real data if available
    if floodpercent_real is not None:
        # define function to get index of nearest observed flood event. 
        def nearest_ind(items, pivot):
            time_diff = np.abs([date - pivot for date in items])
            return time_diff.argmin(0)
        # get nearest observed flood event
        idxA = nearest_ind(time_real, q_tot[0].time.values)
        idxB = nearest_ind(time_real, q_tot[-1].time.values)
        # plot observed flood discharges
        ln3 = ax1.plot(time_real[idxA:idxB+1], floodpercent_real[idxA:idxB+1], color='green', marker='x', markersize=10, linestyle='None', label='Flood Percentage from SAR')
        lns = lns + ln3

    # create legend entries
    labs = [l.get_label() for l in lns]
    ax1.legend(lns, labs)

    plt.setp(ax1.get_xticklabels(), rotation=45)
    
    # save figures
    if saveIt:
        Path(fc.selected,'Figures').mkdir(parents=True, exist_ok=True)
        plt.savefig(Path(fc.selected,'Figures','floodANDdischargeVStime.tif'), format='tiff', dpi=300)

    
    plt.show()

    return fig_perc_dis



# define function that will run the comparison
def forecast2observed_comparison(da_forecast, da_observed, myCmap='gray', myType=[], myDate=[]):

    ### --- Get statistics --- ###
    # Calculate statistics
    overall_accuracy, CSI, precision, recall, FAR, PSS = z_score(da_forecast.values, da_observed.values)
    difference_sum = np.nansum(abs(da_forecast.values - da_observed.values))
    difference_percent = difference_sum / np.count_nonzero(~np.isnan(da_forecast.values)) * 100.0
    # Print statistics
    print(f"Overall accuracy: {overall_accuracy:.2f}%")
    print(f"Critical Success Index: {CSI:.2f}%")
    print(f"Precision: {precision:.2f}%")
    print(f"Recall: {recall:.2f}%")
    print(f"FAR: {FAR:.2f}%")
    print(f"PSS: {PSS:.2f}%")
    print(f"Difference Summation: {difference_sum:.0f}")
    print(f"Difference Percent: {difference_percent:.2f}%")
    info = (f"\n\nOverall accuracy: {overall_accuracy:.2f}% \nCritical Success Index: {CSI:.2f}% \nPrecision: {precision:.2f}%"
            "\nRecall: {recall:.2f}% \nDifference Summation: {difference_sum:.0f} \nDifference Percent: {difference_percent:.2f}%")

    info = f"""
    Overall accuracy: {overall_accuracy:.2f}%
    Critical Success Index: {CSI:.2f}%
    Precision: {precision:.2f}%
    Recall: {recall:.2f}%
    FAR: {FAR:.2f}%
    PSS: {PSS:.2f}%
    Difference Summation: {difference_sum:.0f}
    Difference Percent: {difference_percent:.2f}%
    """
    
    ### --- Plot the forecasts --- ###
    
    # Plot the difference between the forecast and real image
    fig_results, ax_results = plt.subplots(1,3, figsize=(20,9))
    # im = ax_results[0].imshow(np.round(da_forecast.values - da_observed.values), vmin=-1, vmax=1, cmap = 'seismic_r', interpolation=None)
    im = ax_results[0].imshow(da_forecast.values - da_observed.values, vmin=-1, vmax=1, cmap = 'seismic_r', interpolation='none')
    plt.colorbar(im, ax=ax_results[0])
    ax_results[0].set_title('Difference Map\nBlue=extra, Red=missing')
    # Plot the forecast
    # im = ax_results[1].imshow(np.round(da_forecast.values), vmin = -1, vmax = 1, cmap=myCmap, interpolation=None)
    im = ax_results[1].imshow(da_forecast.values, vmin = -1, vmax = 1, cmap=myCmap, interpolation='none')
    plt.colorbar(im, ax = ax_results[1])
    ax_results[1].set_title('Forecast')
    # Plot the real image
    # im = ax_results[2].imshow(np.round(da_observed.values), vmin = -1, vmax = 1, cmap=myCmap, interpolation=None)
    im = ax_results[2].imshow(da_observed.values, vmin = -1, vmax = 1, cmap=myCmap, interpolation='none')
    plt.colorbar(im, ax=ax_results[2])
    ax_results[2].set_title('Real image')
    # # put critical stats info in a 4th subplot for easy access when evaluating data
    # ax_results[3].text(0.0, 0.0, info)
    fig_results.text(1.0, 0.5, info, ha='left', va='center')

    if myDate:
        if 'n' in myType:
            fig_results.suptitle(f'Neural Network: {myDate}')
        elif 'p' in myType:
            fig_results.suptitle(f'Polynomial: {myDate}')
        else:
            fig_results.suptitle(f'{myDate}')

    plt.tight_layout(rect=[0, 0, 0.95, 1])
    
    # display image
    plt.show()
    
    return overall_accuracy, CSI, precision, recall, FAR, PSS, difference_sum, difference_percent, fig_results



# convert water mask dates to datetime64 dates
def get_WMDate(WMflnm, format='none'):
    # parse water mask name
    pathNflnm, file_extension = os.path.splitext(WMflnm)
    pieces_path = pathNflnm.split('/') # the pieces of the path. 
    pieces_flnm = pieces_path[-1].split('_')
    
    if format == 'none':
        daDate = pieces_flnm[0]
        WMDate = np.datetime64(daDate[0:4]+'-'+daDate[4:6]+'-'+daDate[6:]) # save as a datetime64
    elif format == 'dashed':
        daDate = pieces_flnm[-1]
        WMDate = np.datetime64(daDate[0:4]+'-'+daDate[4:6]+'-'+daDate[6:]) # save as a datetime64

    return WMDate


# function to save statistics in a CSV file.
def stats2csv(flnm,header,stats,forecasting_file):
    with open(flnm,'w') as myFile:
        wr = csv.writer(myFile, quoting=csv.QUOTE_ALL)
        wr.writerows([header])
        wr.writerows(stats)
        wr.writerows([['File used for forecasting',forecasting_file]])


def water_thresh(myXarray, myThresh=0.5):
    # myFloatArray - 2D array of float values to be converted based on 'thresh'
    # thresh       - float value separating water from not water
    
    # Create newArray into which to copy myXarray and modify it as needed. 
    newArray = myXarray.copy(deep=True)

    # Convert 254 (or higher) to NaN
    idxs = (newArray.values >= 254)
    newArray.values[idxs] = np.nan
    
    # Convert indices larger than 'myThresh' and less than 254 to 1 (water)
    # Note: 254 is selected as opposed to 255 in order to avoid rounding errors since the xarray is still float. 
    # E.g., float(255) might equal 254.9999999999, which would mean a bunch of 254.### would just be hanging around. 
    idxs = (newArray.values >= myThresh)
    newArray.values[idxs] = 1.0

    # Convert indices between 'myThresh' and 0 (not water)
    idxs = (newArray.values < myThresh)
    newArray.values[idxs] = 0.0


    # myXarray.values = np.where(myXarray.values==255, np.nan, myXarray.values)

    return newArray


### --- Find matching index pairs between water masks and forecasts based on time stamps. --- ###
def get_index_pairs(myDates, myForecasts):
    
    index_pairs = []
    for i, ts1 in enumerate(myDates):
        for j, ts2 in enumerate(myForecasts.time.values.astype('datetime64[D]')):
            if ts1 == ts2:
                index_pairs.append((i,j))
    
    return index_pairs


### --- Get z-scores for each forecast/observation pair and threshold --- ###
def calcAllStats(myThresholds, index_pairs, myForecasts, myWaterMasks):

    nrows, ncols = len(index_pairs), len(myThresholds)
    
    # Initialize containers for z-scores

    overall_accuracy_all = np.zeros((nrows, ncols))
    CSI_all =              np.zeros((nrows, ncols))
    precision_all =        np.zeros((nrows, ncols))
    recall_all =           np.zeros((nrows, ncols))
    FAR_all =              np.zeros((nrows, ncols))
    PSS_all =              np.zeros((nrows, ncols))

    # Loop through each matching forecast/observation pair. 
    for i, pair in enumerate(index_pairs): 
        # Loop through each threshold. 
        for j, myThreshold in enumerate(myThresholds): 
            # apply the water threshold to create a temporary forecast against which the statistics will be compared. 
            forecast_thresh = water_thresh(myForecasts[pair[1]], myThreshold)
            overall_accuracy, CSI, precision, recall, FAR, PSS = z_score(forecast_thresh.values, myWaterMasks[pair[0]].values)
            # put statistics in containers
            overall_accuracy_all[i,j] = overall_accuracy
            CSI_all[i,j] =              CSI
            precision_all[i,j] =        precision
            recall_all[i,j] =           recall
            FAR_all[i,j] =              FAR
            PSS_all[i,j] =              PSS

    return overall_accuracy_all, CSI_all, precision_all, recall_all, FAR_all, PSS_all

def calcSummaryStats(stats_all, myThresholds):
            
    ### --- Calculate statistics (standard deviation, mean, median) for z-scores --- ###
    # initialize containers
    # overall accuracy
    stats_mean = np.zeros(len(myThresholds))
    stats_std = np.zeros(len(myThresholds))
    stats_median = np.zeros(len(myThresholds))
    # calculate summary stats
    for i, myThreshold in enumerate(myThresholds):
        # overall accuracy
        stats_mean[i] = np.mean(stats_all[:,i])
        stats_std[i]  = np.std(stats_all[:,i])
        stats_median[i]  = np.median(stats_all[:,i])

    return stats_mean, stats_std, stats_median


### --- Plot summary statistics of forecasts for various thresholds --- ###
def zScoreEvaluation(myDates, myWaterMasks, myThresholds, myForecasts, myTitle, plotIt=False):
    # function to get best threshold value via a calculation and plot summary zscore info. 
    
    ### --- Find matching index pairs between water masks and forecasts based on time stamps. --- ###
    index_pairs = get_index_pairs(myDates, myForecasts)

    ### --- Get z-scores for each forecast/observation pair and threshold --- ###
    overall_accuracy_all, CSI_all, precision_all, recall_all, FAR_all, PSS_all = calcAllStats(myThresholds, index_pairs, myForecasts, myWaterMasks)
    
    ### --- Calculate statistics (standard deviation, mean, median) for z-scores --- ###
    # overall accuracy
    overall_accuracy_mean, overall_accuracy_std, overall_accuracy_median = calcSummaryStats(overall_accuracy_all, myThresholds)
    # CSI
    CSI_mean, CSI_std, CSI_median = calcSummaryStats(CSI_all, myThresholds)
    # precision
    precision_mean, precision_std, precision_median = calcSummaryStats(precision_all, myThresholds)
    # recall
    recall_mean, recall_std, recall_median = calcSummaryStats(recall_all, myThresholds)
    # FAR
    FAR_mean, FAR_std, FAR_median = calcSummaryStats(FAR_all, myThresholds)
    # PSS
    PSS_mean, PSS_std, PSS_median = calcSummaryStats(PSS_all, myThresholds)
    
    ### --- Plot thresholds vs z-scores --- ###
    if plotIt:
        fig_zScores, ax_zScores = plt.subplots(nrows=3, ncols=2, figsize=(9,12))
        ax_zScores = ax_zScores.flatten()
        # plot mean scores
        ax_zScores[0].plot(myThresholds, overall_accuracy_mean, color='blue')
        ax_zScores[1].plot(myThresholds, CSI_mean, color='blue')
        ax_zScores[2].plot(myThresholds, precision_mean, color='blue')
        ax_zScores[3].plot(myThresholds, recall_mean, color='blue')
        ax_zScores[4].plot(myThresholds, FAR_mean, color='blue')
        ax_zScores[5].plot(myThresholds, PSS_mean, color='blue')    
        # plot mean scores +- std to show range
        ax_zScores[0].plot(myThresholds, overall_accuracy_mean - overall_accuracy_std, marker='^', color='blue', linestyle='None')
        ax_zScores[0].plot(myThresholds, overall_accuracy_mean + overall_accuracy_std, marker='v', color='blue', linestyle='None')
        ax_zScores[1].plot(myThresholds, CSI_mean - CSI_std, marker='^', color='blue', linestyle='None')
        ax_zScores[1].plot(myThresholds, CSI_mean + CSI_std, marker='v', color='blue', linestyle='None')
        ax_zScores[2].plot(myThresholds, precision_mean - precision_std, marker='^', color='blue', linestyle='None')
        ax_zScores[2].plot(myThresholds, precision_mean + precision_std, marker='v', color='blue', linestyle='None')
        ax_zScores[3].plot(myThresholds, recall_mean - recall_std, marker='^', color='blue', linestyle='None')
        ax_zScores[3].plot(myThresholds, recall_mean + recall_std, marker='v', color='blue', linestyle='None')
        ax_zScores[4].plot(myThresholds, FAR_mean - FAR_std, marker='^', color='blue', linestyle='None')
        ax_zScores[4].plot(myThresholds, FAR_mean + FAR_std, marker='v', color='blue', linestyle='None')
        ax_zScores[5].plot(myThresholds, PSS_mean - PSS_std, marker='^', color='blue', linestyle='None')
        ax_zScores[5].plot(myThresholds, PSS_mean + PSS_std, marker='v', color='blue', linestyle='None')    
        # plot median scores
        ax_zScores[0].plot(myThresholds, overall_accuracy_median, color='green')
        ax_zScores[1].plot(myThresholds, CSI_median, color='green')
        ax_zScores[2].plot(myThresholds, precision_median, color='green')
        ax_zScores[3].plot(myThresholds, recall_median, color='green')
        ax_zScores[4].plot(myThresholds, FAR_median, color='green')
        ax_zScores[5].plot(myThresholds, PSS_median, color='green')
        # plot an 'x' on the highest PSS score and give the threshold value. 
        myX, myY = myThresholds[ np.argmax(PSS_median) ], PSS_median[ np.argmax(PSS_median) ]
        ax_zScores[5].plot(myX, myY, color='red', marker='x')
        text_label = f"x={myThresholds[ np.argmax(PSS_median) ]}"
        ax_zScores[5].annotate(text_label, 
                               xy = (myX, myY), 
                               xytext = (myX+0.05, myY-30.0), 
                               arrowprops=dict(facecolor='black', shrink=0.15)
                              )    
        # add labels to x and y axes
        for ax in ax_zScores:
            ax.set_xlabel('Threshold Values')
            ax.set_ylabel('Percent')
        # add grid to all subplots
        ax_zScores[0].grid()
        ax_zScores[1].grid()
        ax_zScores[2].grid()
        ax_zScores[3].grid()
        ax_zScores[4].grid()
        ax_zScores[5].grid()
        # set titles
        ax_zScores[0].set_title('Overall Accuracy')
        ax_zScores[1].set_title('CSI')
        ax_zScores[2].set_title('Precision')
        ax_zScores[3].set_title('Recall')
        ax_zScores[4].set_title('False Alarm Rate (FAR)')
        ax_zScores[5].set_title('Pierce Skill Score (PSS)')
    
        fig_zScores.suptitle(myTitle)
        
        plt.tight_layout()
        plt.show()
    
    return myThresholds[ np.argmax(PSS_median) ]

***
# 5. Select Forecasting Date Range

## 5.1 Run a Hindcast Starting Using Validation Set

Let's identify the end date of the dataset used for model training.

In [ ]:
print(f'End Date for Training Set: {da_tot[ind_flood].time.values.astype("datetime64[D]")}')

Now define forecast start date relative to end of training set.

In [ ]:
# input year, month, and day
myDate = da_tot[ind_flood].time.values.astype('datetime64[D]') - 24

numDays = 200 # number of days to forecast. 

In [ ]:
# Find the index of the first day of the forecast
ind_start = np.where(q_tot.time == myDate)[0][0]

# Find the index of the last day of the forecast
ind_stop = np.where(q_tot.time == myDate)[0][0]+numDays

# Crop the discharge to the same range as the forecast
q_forecast = q_tot.isel(time=slice(ind_start, ind_stop + 1))

Validate forecasting timeframe.

In [ ]:
print(f"Start Date: {q_forecast[0].time.values.astype('datetime64[D]')}")
print(f"End Date:   {q_forecast[-1].time.values.astype('datetime64[D]')}")
print(f"All Dates:  \n{q_forecast.time.values.astype('datetime64[D]')}")

## 5.2 Option 2: Generalized forecast

This will acquire the 15 day forecast of geoglows data and forecast from today to 15 days from now. 

In [ ]:
ds = geoglows.data.forecast(ReachID, format='xarray')
ds = ds.assign_coords(time=pd.to_datetime(ds.time.values))
ds = ds.resample(time='1D').sum()

In [ ]:
q_forecast = ds['flow_median'].rename('discharge')

## 5.3 Option 3: Generalized hindcast

In this, a 'forecast' is made for the 12 days including and prior to the desired flooding event to predict. This is generally used to assess the accuracy and precision of a given forecast model. 

In [ ]:
### --- Prepare Discharge dataset --- ###

# Find the index of the first day of the forecast
ind_start = np.where(q_tot.time == q.time[-1])[0][0]

# Find the index of the last day of the forecast
# ind_stop = np.where(q_tot.time == da_tot.time[ind_flood])[0][0]
ind_stop = np.where(q_tot.time == da_tot.time)[0][0]
ind_stop = ind_start + 15

# Crop the discharge to the same range as the forecast
q_forecast = q_tot.isel(time=slice(ind_start, ind_stop + 1)).resample(time='1D').mean()#.rolling(time=5).mean()
q_forecast

***
# 6. Now Run the Forecast
## 6.1 Run Each Mode Forecast with Best Performing Mode

These cells will calculate the unique signal from each mode of interest. 

In [ ]:
### --- User selects modes to combine --- ###

print(f'Available Modes:\nmodes_nn:   {modes_nn}\nmodes_poly: {modes_poly}')

In [ ]:
### --- Select modes to run --- ###
# It is suggested that all modes are run so they can be combined as desired by users in the next sub-section. 

# Select which modes to include for the polynomial forecast
myModes_poly = modes_poly
# myModes_poly = [1, 2,]

# Select which modes to include for the neural network forecast
myModes_nn = modes_nn
# myModes_nn = [1, 2,]

In [ ]:
### --- Initialize containers --- ###
# get the number of modes, # of time entires, and spatial size (n_modes, t, y, x) of forecasts_noCenter
n_modes, y, x = reof_ds.spatial_modes.shape
t = q_forecast.shape[0]
# initialize forecasts_noCenter
forecasts_noCenter_nn = np.zeros((n_modes, t, y, x))
forecasts_noCenter_poly = np.zeros((n_modes, t, y, x))

In [ ]:
# loop through calculations and store them
# Calculate unique signal using neural network models. 
for myMode in myModes_nn:
    forecasts_noCenter_nn[myMode-1,:,:,:] = partialForecastCalc("neural network", myMode, myModes_nn, q_forecast, reof_ds, models_nn)

# Calculate unique signal using polynomial models. 
for myMode in myModes_poly: 
    forecasts_noCenter_poly[myMode-1,:,:,:] = partialForecastCalc("polynomial", myMode, myModes_poly, q_forecast, reof_ds, polys)

***
## 6.2 Combine modes as desired

The below cells create the single-mode and multi-mode forecasts. By changing [myCombo_nn] and [myCombo_poly], users can re-calculate multi-mode forecasts with any of the partially calculated forecasts from above on the fly, rather than having to redo the full forecast every time they want a different set of modes. This significantly reduces computation time for new multi-mode sets. 

WARNING!!! The new multi-mode set up allows for great modularity, but comes at the price of using significant RAM storage. If allocated RAM is exceeded, a "Server unavailable or unreachable" error will pop-up. 

In [ ]:
# # Running this code can help keep RAM usage down when trying different combinations of modes
try: 
    del da_nn_multi, da_poly_multi, da_nn_single, da_poly_single
    print("Holdover variables deleted for a new run.")
except: 
    print("No holdover variables to delete.")

In [ ]:
### --- Single highest mode calcualtion --- ###

da_nn_single   = finalForecastCalc(forecasts_noCenter_nn, reof_ds, 1)
da_poly_single = finalForecastCalc(forecasts_noCenter_poly, reof_ds, 1)

In [ ]:
### --- Select modes for multi-mode calculation --- ###

myCombo_nn   = [1, 2,] # myModes_nn
myCombo_poly = [1, 2,] # myModes_poly
if len(myCombo_nn) > len(myModes_nn):
    raise ValueError("The size of 'myCombo_nn' exceeds the number of available modes.")
elif len(myCombo_poly) > len(myModes_poly): 
    raise ValueError("The size of 'myCombo_poly' exceeds the number of available modes.")

print(f"myCombo_nn   = {myCombo_nn}")
print(f"myCombo_poly = {myCombo_poly}")

In [ ]:
### --- Multi-mode calculation --- ###

da_nn_multi   = finalForecastCalc(forecasts_noCenter_nn, reof_ds, myCombo_nn)
da_poly_multi = finalForecastCalc(forecasts_noCenter_poly, reof_ds, myCombo_poly)

***
## 6.3 Threshold Forecast to Water Map

### 6.3.1 Select value for water threshold (OPTIONAL)

The forecasts are currently in a float format and must be thresholded. The default threshold value is 0.5; any pixel in the forecast below this value will be considered dry land, anything above water. This generally gives the best results. The below code is used for exploring the best value to use as the water threshold. 

### Z-scores through time

The following code calculates a number of metrics to assess the fidelity of the I will calculate the z-scores for each forecast/observation pair and then plot these against threshold values. Use this code when it is important for all of the forecasts to be as accurate as possible. 

In [ ]:
### --- INITIALIZE THRESHOLDS --- ###
thresholds = np.linspace(0.0, 1.0, 21)
print(f"Selected Threshold Values: \n{thresholds}")

In [ ]:
### --- Load water masks (AKA, the observations) --- ###

# Designate important file loading parameters. 
myFormat = 'none'
myPattern = r'\d{8}_water_mask_combined.tif*'

# Get file names for water masks. 
WMs = [f for f in os.listdir(Path(fc.selected,'Water_Masks')) if re.match(myPattern, f)]
WMs.sort()

# Get dates for each water mask. 
WMs_dates = [get_WMDate(WM, format=myFormat) for WM in WMs]
WMs_dates.sort()

# Create time dimension for water masks container. 
myTimes = pd.DatetimeIndex(WMs_dates)
myTimes.name = "time"
# Load the water masks into an xarray. 
water_masks = xr.concat([rxr.open_rasterio(Path(fc.selected, 'Water_Masks', flnm)).squeeze(dim='band') 
                         for flnm in WMs], dim=myTimes)
# drop unnecessary variables from the xarray. 
water_masks = water_masks.drop_vars(['band', 'spatial_ref']) 

In [ ]:
# Plot metrics
thresh_nn_sM = zScoreEvaluation(WMs_dates, water_masks, thresholds, da_nn_single, 'Neural Network\nSingle Mode', True)
thresh_nn_mM = zScoreEvaluation(WMs_dates, water_masks, thresholds, da_nn_multi, 'Neural Network\nMulti Mode', True)
thresh_poly_sM = zScoreEvaluation(WMs_dates, water_masks, thresholds, da_poly_single, 'Polynomial\nSingle Mode', True)
thresh_poly_mM = zScoreEvaluation(WMs_dates, water_masks, thresholds, da_poly_multi, 'Polynomial\nMulti Mode', True)

***
### Contour Mapping of Flooded Regions

The following code plots contours of the forecasted flooding for different thresholds onto maps of observed flooding. Use this code to determine the best accuracy for a particular flood. 

In [ ]:
### --- Create binning --- ###

# Set 'myBins' parameters. 
bin_min  = 0.3
bin_max  = 0.7
bin_step = 0.1

# Create bins
myBins = np.arange(bin_min, bin_max+bin_step, bin_step)


forecast = da_nn_single[0].values
# certaintyMap = copy.deepcopy(forecast) - 0.5
certaintyMap = copy.deepcopy(da_nn_single[0].values)

### Load water mask and make sure indices are in the correct place

In [ ]:
### --- Load water masks (AKA, the observations) --- ###

# Designate important file loading parameters. 
myFormat = 'none'
myPattern = r'\d{8}_water_mask_combined.tif*'

# Get file names for water masks. 
WMs = [f for f in os.listdir(Path(fc.selected,'Water_Masks')) if re.match(myPattern, f)]
WMs.sort()

# Get dates for each water mask. 
WMs_dates = [get_WMDate(WM, format=myFormat) for WM in WMs]
WMs_dates.sort()

# Create time dimension for water masks container. 
myTimes = pd.DatetimeIndex(WMs_dates)
myTimes.name = "time"
# Load the water masks into an xarray. 
water_masks = xr.concat([rxr.open_rasterio(Path(fc.selected, 'Water_Masks', flnm)).squeeze(dim='band') 
                         for flnm in WMs], dim=myTimes)
# drop unnecessary variables from the xarray. 
water_masks = water_masks.drop_vars(['band', 'spatial_ref']) 

In [ ]:
index_pairs = get_index_pairs(WMs_dates, da_nn_single)

### Find indices

Find indices of forecast that are water (by each bin) in the observed water. 

This should produce a 2 column vector. Column 1 contains the percentage of that forecast bin that truly is water (true positive); Column 2 contains the percentage of the forecast that is not observed as water. 

In [ ]:
### --- Put indices for each bin into the container --- ###
# Initialize container for indices. This is a dictionary. 
myBinIndices = {}
# Initialize list that will contain the keys for my indices container. Useful for later reference. 
myKeys = []

# Loop through each bin and store indices. 
for i in range(0,len(myBins)-1):
    # Get indices for the ith bin. 
    row_indices, col_indices = np.where( (certaintyMap >= myBins[i]) & (certaintyMap <= myBins[i+1]) )

    # Create key with only 1 decimal place. 
    myKey = f"{myBins[i]:.1f}-{myBins[i+1]:.1f}"
    myKeys.append(myKey)
    # store indices in dictionary. 
    myBinIndices[myKey] = np.vstack((row_indices, col_indices)).T

In [ ]:
def get_indicesFromBins(theIndices, theBins):
    # This function returns the list of all indices from 'theBin' upwards. 
    
    # theIndices - xarray of bins containing the indices. 
    # theBins - the bins to sum together. 

    # Initialize container to hold indice pairs.
    allIndices = []
    tempIndices = []
    
    for bin in theBins: 
        # get the xs and ys. 
        xs, ys = theIndices[bin][:,0], theIndices[bin][:,1]
        # put the indices in list form and add to container. 
        allIndices.extend(list(zip(xs,ys)))
    
    return allIndices

### Plot water masks and overlay contours of thresholded forecasts

In [ ]:
### --- Plot water mask with forecast based contour for each matching day --- ###

# The below code is meant to plot the water mask with the forecast based contouring for the different thresholds. 
# I should end up with as many plots as there are water masks and forecasts with matching dates. 
import matplotlib.colors


for pair in index_pairs:


    forecast = copy.deepcopy(da_nn_single[pair[1]])
    certaintyMap = forecast.values
    
    # print(f"forecast date:   {forecast.time.values.astype('datetime64[D]')}")
    # print(f"water mask date: {water_masks[pair[0]].time.values.astype('datetime64[D]')}")

    ### --- Get bins and indices for the ith forecast. --- ###
    # Initialize container for indices. This is a dictionary. 
    myBinIndices = {}
    # Initialize list that will contain the keys for my indices container. Useful for later reference. 
    myKeys = []
    
    # Loop through each bin and store indices. 
    for i in range(0,len(myBins)-1):
        # Get indices for the ith bin. 
        row_indices, col_indices = np.where( (certaintyMap >= myBins[i]) & (certaintyMap <= myBins[i+1]) )
    
        # Create key with only 1 decimal place. 
        myKey = f"{myBins[i]:.1f}-{myBins[i+1]:.1f}"
        myKeys.append(myKey)
        # store indices in dictionary. 
        myBinIndices[myKey] = np.vstack((row_indices, col_indices)).T

    
    # Plot hte water mask
    myFig, myAxes = plt.subplots(1,1, figsize=(10,10))
    myAxes.imshow(water_masks[ pair[0] ].values, vmin=-1, vmax=1, cmap='seismic_r', interpolation='none')
    
    # set up colors for contours. 
    vals = np.linspace(0, 1, len(myBinIndices))
    
    waterContour = np.zeros( water_masks[ pair[0] ].values.shape )
    
    for j, bin in enumerate(myBinIndices): 
    
        # plot water points as contour maps. 
        # Initialize array of zeros. 
        # Get indices of water for the current bin.     
        keyNum = myKeys.index(bin)
        theBins = myKeys[keyNum:]
        allIndices = get_indicesFromBins(myBinIndices, theBins)
        ys, xs = [y for y,_ in allIndices], [x for _,x in allIndices]
        # Apply indices of water to 'waterContour'. 
        # waterContour[ys, xs] = j
        waterContour[ys, xs] = myBins[j]
        
    # Plot contour map. 
    cf = myAxes.contour(waterContour, linewidths=0.5, cmap='autumn', levels=myBins)
    myAxes.clabel(cf, inline=True, fontsize=10)
    myAxes.clabel(cf, inline=True, fontsize=20)
    myAxes.set_title(f"Date: {forecast.time.values.astype('datetime64[D]')}")
    
    # Create filled in colorbar. 
    norm = matplotlib.colors.Normalize(vmin=bin_min, vmax=bin_max)
    sm = plt.cm.ScalarMappable(norm=norm, cmap=cf.cmap)
    sm.set_array([])
    
    cbar = myFig.colorbar(sm, ax=myAxes, ticks=cf.levels)
    # cbar.set_ticks(myBins[1:])
    # cbar.set_ticklabels(myBins[1:])
    
    plt.show()
    

### 6.3.2 Threshold to water map

The above forecasts are float. This is similar to (though distinct from) a probability map. The higher the value, the more likely it is to flood there; conversely, the lower the value, the less likely it is to flood at that location. 

The below code will use a simple threshold to convert from "probabilities" (float) to a binary-like water/not-water map (0 for not water, 1 for water, and 255 or NaN for regions outside of the AOI). 

In [ ]:
print("Thresholds calculated based on Pierce Skill Score (PSS):")
print(f"   NN single Mode:   {thresh_nn_sM}")
print(f"   NN multi Mode:    {thresh_nn_mM}")
print(f"   Poly single Mode: {thresh_poly_sM}")
print(f"   Poly multi Mode:  {thresh_poly_mM}")

In [ ]:
customThreshold = True

In [ ]:
if customThreshold: 
    # use the custom thresholds determined above. 
    print("Using custom thresholds...")
    da_nn_single   = water_thresh(da_nn_single, thresh_nn_sM)
    da_nn_multi    = water_thresh(da_nn_multi, thresh_nn_mM)
    da_poly_single = water_thresh(da_poly_single, thresh_poly_sM)
    da_poly_multi  = water_thresh(da_poly_multi, thresh_poly_mM)

else:
    # use the default threshold value of 0.5. 
    print("Using default threshold of 0.5...")
    da_nn_single   = water_thresh(da_nn_single, thresh)
    da_nn_multi    = water_thresh(da_nn_multi, thresh)
    da_poly_single = water_thresh(da_poly_single, thresh)
    da_poly_multi  = water_thresh(da_poly_multi, thresh)

print("Thresholding complete!")

### 6.3.3 Ensure Forecasting Run Completed Successfully 

The only unique values should be 0 (not water), 1 (water), and NaN (regions outside the region of interest).

Note: This is only a valid check  if the water forecasts are thresholded. Without thresholding, there will be many different floating point values. 

In [ ]:
unique_elements_poly, counts_poly = np.unique(da_poly_single.values, return_counts=True)
unique_elements_nn,   counts_nn    = np.unique(da_nn_single.values  , return_counts=True)
print(f"Polynomial    : {unique_elements_poly}")
print(f"Neural Network: {unique_elements_nn  }")
print(f"counts_poly : {counts_poly}")
print(f"counts_nn   : {counts_nn  }")

In [ ]:
unique_elements_poly, counts_poly = np.unique(da_poly_multi.values, return_counts=True)
unique_elements_nn,   counts_nn    = np.unique(da_nn_multi.values  , return_counts=True)
print(f"Polynomial    : {unique_elements_poly}")
print(f"Neural Network: {unique_elements_nn  }")
print(f"counts_poly : {counts_poly}")
print(f"counts_nn   : {counts_nn  }")

***
## 6.4 Clip Forecasts to Relevant Region of Interest (ROI) (i.e., remove the buffer zone) (OPTIONAL)

The REOF suffers from edge effects. Edge effects reduce the accuracy and precision of forecasts. Because of this, it is a part of best practices to run the training on a region with a buffer zone which is then later removed. The required buffer zone is usually a few kilometers. 

This section of code clips the forecasts to the ROI. The ROI is usually defined as a watershed. 

In [ ]:
from osgeo import gdal
gdal.UseExceptions()
import glob

In [ ]:
# Write function to save the forecasts as geotiffs. 
def saveForecasts_v2(saveName, tiff, CRS, transform):
    # make directory if necessary
    directory = os.path.dirname(saveName)
    if not os.path.exists(directory):
        os.makedirs(directory)
    else:
        # print(f"Directory already exists: {directory}")
        pass
    
    myValues = np.abs(tiff.values) # need to take the absolute value as there are some "negative" zero values (I know, negative zero isn't actually a thing; the computer sure thinks it is)
    # open raster file and save
    with rasterio.open(
        saveName,
        mode="w",
        driver="GTiff",
        height=tiff.shape[0],
        width=tiff.shape[1],
        count=1,
        dtype=tiff.dtype,
        crs=CRS,
        transform=transform,
    ) as new_dataset:
        new_dataset.write(myValues, 1)

    return
    

In [ ]:
# Create debuffered forecasts and water masks. 

if debuff:     
    # Initialize important bits
    # get the shapefile info. 
    debuff_shapefile.selected
    shp = Path(debuff_shapefile.selected)
    if shp.suffix == '.shp':
        gInfo = gdal.OpenEx(str(shp))
        layer = gInfo.GetLayer()
        feature = layer.GetFeature(0)
        wkt = feature.GetGeometryRef().ExportToWkt()
    # set up folder names in which to save the files
    temp_folder = Path(fc.selected, 'debuffing')
    folder_debuffed = Path(fc.selected, 'debuffed')

    # create full list of debuffered files
    debuffeds_nn_sM   = []
    debuffeds_nn_mM   = []
    debuffeds_poly_sM = []
    debuffeds_poly_mM = []
    
    ### --- Subset datacheck, forecasts, and water masks --- ### 
    # First, create new datacheck and forecasts that are subsetted to the relevant region of interest. 
    os.makedirs(temp_folder, exist_ok=True)
    os.makedirs(folder_debuffed, exist_ok=True)
    # save the data check (da_tot)
    saveName_dataCheck = Path(temp_folder,'dataCheck.tif')
    saveForecasts_v2( saveName_dataCheck, da_tot, CRS, transform)
    debuffed_dataCheck = Path(folder_debuffed,'dataCheck.tif')
    gdal.Warp(str(debuffed_dataCheck), str(saveName_dataCheck), dstNodata=np.nan, cutlineDSName=str(shp), cropToCutline=True)
    # Subset the relevant files (the forecasts, and water masks)
    # subset the forecasts
    county = 0
    for tiff_nn_sM, tiff_nn_mM, tiff_poly_sM, tiff_poly_mM in zip(da_nn_single, da_nn_multi, da_poly_single, da_poly_multi):
        # create save name for 'debuffering' files
        fileName=f'Day-{county:03d}_{str(tiff_nn_sM.time.dt.strftime("%Y%m%d").values)}.tif'
        # print(fileName)
        # current full save name for 'debuffering' files
        saveName_nn_sM   = Path(temp_folder, 'nn_sM',   fileName)
        saveName_nn_mM   = Path(temp_folder, 'nn_mM',   fileName)
        saveName_poly_sM = Path(temp_folder, 'poly_sM', fileName)
        saveName_poly_mM = Path(temp_folder, 'poly_mM', fileName)
        saveForecasts_v2(saveName_nn_sM,   tiff_nn_sM,   CRS, transform)
        saveForecasts_v2(saveName_nn_mM,   tiff_nn_mM,   CRS, transform)
        saveForecasts_v2(saveName_poly_sM, tiff_poly_sM, CRS, transform)
        saveForecasts_v2(saveName_poly_mM, tiff_poly_mM, CRS, transform)
        # Subset 'debuffering' files and save them in 'debuffed'
        # set up file names
        debuffed_nn_sM   = Path(folder_debuffed, 'nn_sM',   fileName)
        debuffed_nn_mM   = Path(folder_debuffed, 'nn_mM',   fileName)
        debuffed_poly_sM = Path(folder_debuffed, 'poly_sM', fileName)
        debuffed_poly_mM = Path(folder_debuffed, 'poly_mM', fileName)
        
        debuffeds_nn_sM.append(  debuffed_nn_sM)
        debuffeds_nn_mM.append(  debuffed_nn_mM)
        debuffeds_poly_sM.append(debuffed_poly_sM)
        debuffeds_poly_mM.append(debuffed_poly_mM)
        # make directory if it doesn't exist
        if not os.path.exists( os.path.dirname(debuffed_nn_sM) ):
            os.makedirs( os.path.dirname(debuffed_nn_sM) )
        if not os.path.exists( os.path.dirname(debuffed_nn_mM) ):
            os.makedirs( os.path.dirname(debuffed_nn_mM) )
        if not os.path.exists( os.path.dirname(debuffed_poly_sM) ):
            os.makedirs( os.path.dirname(debuffed_poly_sM) )
        if not os.path.exists( os.path.dirname(debuffed_poly_mM) ):
            os.makedirs( os.path.dirname(debuffed_poly_mM) )
        # use gdal to create subsets
        gdal.Warp(str(debuffed_nn_sM),   str(saveName_nn_sM),   dstNodata=np.nan, cutlineDSName=str(shp), cropToCutline=True)
        gdal.Warp(str(debuffed_nn_mM),   str(saveName_nn_mM),   dstNodata=np.nan, cutlineDSName=str(shp), cropToCutline=True)
        gdal.Warp(str(debuffed_poly_sM), str(saveName_poly_sM), dstNodata=np.nan, cutlineDSName=str(shp), cropToCutline=True)
        gdal.Warp(str(debuffed_poly_mM), str(saveName_poly_mM), dstNodata=np.nan, cutlineDSName=str(shp), cropToCutline=True)
        county = county + 1
    # subset the water masks
    WMs = glob.glob(fc.selected + 'Water_Masks/*combined.tif*')
    WMs.sort()
    WMDates = []
    for WM in WMs:
        WMDate = get_WMDate(WM)
        WMDates.append(WMDate)
        debuffed_WM = Path(folder_debuffed, 'Water_Masks', str(WMDate).replace("-","")+'_water_mask_combined.tif')
        if not os.path.exists( os.path.dirname(debuffed_WM) ):
            os.makedirs( os.path.dirname(debuffed_WM) )
        gdal.Warp(str(debuffed_WM), str(WM), dstNodata=np.nan, cutlineDSName=str(shp), cropToCutline=True)

    # delete the temporary folder to save space. 
    import shutil
    shutil.rmtree(temp_folder)

In [ ]:
### --- Get Dates from filenames --- ###
def get_dates(dir_path):
    dates = []
    pths = list(dir_path.glob(f'*.tif*'))

    for p in pths:
        date_regex = r'\d{8}'
        date = re.search(date_regex, str(p))
        if date:
            dates.append(date.group(0))
    return dates





### --- Load Geotiffs Function --- ###
def load_tiffs(folder, stop_ind = -1):

    # Gather names of files corresponding to the file type and polarization we want
    tiff_dir = Path(folder)
    tiffs = list(tiff_dir.glob(f'*.tif*'))
    
    # Gather the date of each file
    times = get_dates(tiff_dir)
    
    # Create a list of indices based on the sorted order of times
    sorted_indices = sorted(range(len(times)), key=lambda i: times[i])
    
    # Sort the paths based on the times
    tiffs = [tiffs[i] for i in sorted_indices]
    
    # Sort the times axisdd
    times.sort()
    times = pd.DatetimeIndex(times)
    times.name = "time"
    
    # Create the dataset gathering the input images
    da = xr.concat([rxr.open_rasterio(f).squeeze(dim='band') for f in tiffs[:stop_ind]], dim=times[:stop_ind])
    da = da.drop_vars(['band', 'spatial_ref'])

    print(f"Dataset size in memory: {da.nbytes / 1e6:.2f} MB")

    return da, tiffs, times

In [ ]:
# load debuffered forecasts and water masks

if debuff:
    # load subsetted dataCheck in relevant object
    # da_tot_check = xr.concat([rxr.open_rasterio(dataCheck).squeeze(dim='band') for dataCheck in debuffed_dataCheck, dim=da_tot.time)
    da_tot_check = rxr.open_rasterio(debuffed_dataCheck) # THIS partially WORKS; loads data, but doesn't load time
    da_tot_check = da_tot_check.drop_vars(['band', 'spatial_ref'])
    da_tot_check = da_tot_check[0]
    da_tot_check['time'] = da_tot.time
    # delete original and replace it
    del da_tot
    da_tot = da_tot_check.copy(deep=True)
    
    # Load subsetted forecast values into temporary objects
    # create directory names
    nn_sM_dir   = Path(folder_debuffed, 'nn_sM')
    nn_mM_dir   = Path(folder_debuffed, 'nn_mM')
    poly_sM_dir = Path(folder_debuffed, 'poly_sM')
    poly_mM_dir = Path(folder_debuffed, 'poly_mM')
    # load subsetted forecasts 
    da_nn_sM_check  , _, _ = load_tiffs( nn_sM_dir )
    print('Debuffered single mode neural network forecasts loaded.')
    da_nn_mM_check  , _, _ = load_tiffs( nn_mM_dir )
    print('Debuffered multi mode neural network forecasts loaded.')
    da_poly_sM_check, _, _ = load_tiffs( poly_sM_dir )
    print('Debuffered single mode polynomial forecasts loaded.')
    da_poly_mM_check, _, _ = load_tiffs( poly_mM_dir )
    print('Debuffered multi mode polynomial forecasts loaded.')
    
    # Reassign subsetted forecasts (new, temporary objects) to appropriate variables
    # delete original objects and replace them (this is so they keep the same name and I don't have to change a bunch of other code)
    del da_nn_single, da_nn_multi, da_poly_single, da_poly_multi
    da_nn_single   = da_nn_sM_check.copy(deep=True)
    da_nn_multi    = da_nn_mM_check.copy(deep=True)
    da_poly_single = da_poly_sM_check.copy(deep=True)
    da_poly_multi  = da_poly_mM_check.copy(deep=True)
    
    # delete temporary variables
    del da_tot_check, da_nn_sM_check, da_nn_mM_check, da_poly_sM_check, da_poly_mM_check

***
# 7. Evaluate Accuracy of Forecasts

Compare the forecast results to actual flood extent or SAR imagery.

Note: This is for ONLY the flood chosen in in the previous notebook. The next visualization will compare produced forecasts to all extant water maps. 


In [ ]:
# This is to make visualizations more easily comparable. 
indices = (da_tot >= 255)
da_tot.values[indices] = 0

## 7.1 Compare Long-Term Forecast

This section of code compares the long-term forecast (several months to years worth of forecasts) to historical SAR-derived water extents. 

In [ ]:
# New format
# myFormat = 'dashed'
# myPattern = r'water_extent_\d{8}.tif*'
# Legacy format (before year 2025)
myFormat = 'none'
myPattern = r'\d{8}_water_mask_combined.tif*'

WMs = [f for f in os.listdir(Path(fc.selected,'Water_Masks')) if re.match(myPattern, f)]
WMs.sort()

In [ ]:
### --- Single Mode --- ###
myColorScheme = 'seismic_r' #'gray' # 'seismic'

statistics_singleMode_poly = []
statistics_singleMode_nn   = []
statistics_header = ['Date', 'Overall Accuracy', 'Precision', 'CSI', 'Recall', 'FAR', 'PSS', 'Difference Summation', 'Difference Percentile']
myFigs_singleMode_nn = []
myFigs_singleMode_poly = []
for WM in WMs:    
    WMDate = get_WMDate(WM, format=myFormat)
    almostIdx = np.where(da_poly_single.time.values.astype('datetime64[D]')==WMDate.astype('datetime64[D]'))

    if len(almostIdx[0])==0: 
        pass
    else: 
        # Get index of forecasts that matches the water mask
        idx = almostIdx[0][0]
    
        # Load water mask
        water_mask = rxr.open_rasterio(Path(fc.selected, 'Water_Masks',WM))
        water_mask = water_mask.drop_vars(['band', 'spatial_ref'])                               
        water_mask = water_mask[0]
        indices = (water_mask >= 255)
        water_mask.values[indices] = 0
    
        # Run comparison between forecast and water mask
        overall_accuracy, CSI, precision, recall, FAR, PSS, difference_sum, difference_percent, myFig_singleMode_nn = forecast2observed_comparison(da_nn_single[idx], water_mask, myColorScheme, 'nn', WMDate)
        statistics_singleMode_nn.append([str(WMDate), overall_accuracy, precision, CSI, recall, FAR, PSS, difference_sum, difference_percent])
        myFigs_singleMode_nn.append(myFig_singleMode_nn)
        
        overall_accuracy, CSI, precision, recall, FAR, PSS, difference_sum, difference_percent, myFig_singleMode_poly = forecast2observed_comparison(da_poly_single[idx], water_mask, myColorScheme, 'poly', WMDate)
        statistics_singleMode_poly.append([str(WMDate), overall_accuracy, precision, CSI, recall, FAR, PSS, difference_sum, difference_percent])
        myFigs_singleMode_poly.append(myFig_singleMode_poly)


In [ ]:
### --- Multi Mode --- ###
myColorScheme = 'seismic_r'

statistics_multiMode_poly = []
statistics_multiMode_nn   = []
statistics_header = ['Date', 'Overall Accuracy', 'Precision', 'CSI', 'Recall', 'FAR', 'PSS', 'Difference Summation', 'Difference Percentile']
myFigs_multiMode_nn = []
myFigs_multiMode_poly = []
for WM in WMs:
    WMDate = get_WMDate(WM, format=myFormat)
    almostIdx = np.where(da_poly_multi.time.values.astype('datetime64[D]')==WMDate.astype('datetime64[D]'))

    if len(almostIdx[0])==0: 
        pass
    else: 
        # Get index of forecasts that matches the water mask
        idx = almostIdx[0][0]

        # Load water mask
        water_mask = rxr.open_rasterio(Path(fc.selected, 'Water_Masks',WM))
        water_mask = water_mask.drop_vars(['band', 'spatial_ref'])                               
        water_mask = water_mask[0]
        indices = (water_mask >= 255)
        water_mask.values[indices] = 0
    
        # Run comparison between forecast and water mask
        overall_accuracy, CSI, precision, recall, FAR, PSS, difference_sum, difference_percent, myFig_multiMode_nn = forecast2observed_comparison(da_nn_multi[idx], water_mask, myColorScheme, 'nn', WMDate)
        statistics_multiMode_nn.append([str(WMDate), overall_accuracy, precision, CSI, recall, FAR, PSS, difference_sum, difference_percent])
        myFigs_multiMode_nn.append(myFig_multiMode_nn)
        
        overall_accuracy, CSI, precision, recall, FAR, PSS, difference_sum, difference_percent, myFig_multiMode_poly = forecast2observed_comparison(da_poly_multi[idx], water_mask, myColorScheme, 'poly', WMDate)
        statistics_multiMode_poly.append([str(WMDate), overall_accuracy, precision, CSI, recall, FAR, PSS, difference_sum, difference_percent])
        myFigs_multiMode_poly.append(myFig_multiMode_poly)


***
## 7.2 Plot of Flood Forecasts, Flood Percentages, and Discharge through Time

In [ ]:
# get the flood percentages for the single mode
fp_nn_single   = floodPercentCalc(da_nn_single.values)
fp_poly_single = floodPercentCalc(da_poly_single.values)

In [ ]:
# get the flood percentages for the multi-mode
fp_nn_multi    = floodPercentCalc(da_nn_multi.values)
fp_poly_multi  = floodPercentCalc(da_poly_multi.values)

In [ ]:
fp_axis_lims = [0, np.max(floodpercent)+5]

In [ ]:
# Neural Network Fit
myFldPcnt_singleMode_nn = plot_flood_and_discharge(da_nn_single, fp_nn_single, q_forecast, floodpercent, time_flood, saveIt=False, fp_lims = fp_axis_lims)
myFldPcnt_multiMode_nn = plot_flood_and_discharge(da_nn_multi, fp_nn_multi, q_forecast, floodpercent, time_flood, fp_lims = fp_axis_lims)

In [ ]:
# Polynomial Fit
myFldPcnt_singleMode_poly = plot_flood_and_discharge(da_poly_single, fp_poly_single, q_forecast, floodpercent, time_flood, saveIt=False, fp_lims = fp_axis_lims)
myFldPcnt_multiMode_poly = plot_flood_and_discharge(da_poly_multi, fp_poly_multi, q_forecast, floodpercent, time_flood, saveIt=False, fp_lims = fp_axis_lims)

***
## 7.3 Plot Forecasted versus Observed Flood Percentages

The below code will produce a graph of Forecasted (x-axis) vs Observed (y-axis) flood percentages for each of the single/multi and neural network/polynomial fits. 

A black line will mark the ideal match (i.e., a perfect one). Markers to the right mean the forecast overpredicts water extent; to the left means an underprediction. 

In [ ]:
def realVSobservedFloodPercentages(time_forecasts, time_real, floodPercents_real, myFloodPercents):

    forRMSE = []

    num_rows  = int(len(myFloodPercents) / 2) # note: this assumes there are 4 key/value pairs in the dictionary 'myForecasts'
    num_cols  = 2

    # Create subplots. 
    fig, axes = plt.subplots(num_rows, num_cols, figsize=(9,9))

    # Flatten axes array for simplier indexing
    axes = axes.flatten()

    myKeys = list(myFloodPercents.keys())

    max_fP = 0.0

    i = 0
    for key in myFloodPercents:
        ax = axes[i]
        for j in range(0,len(myFloodPercents[key])):
            # Get index for matching dates of floods. 
            try: 
                idx = np.where(time_forecasts[j] == time_real)[0][0]
            except: 
                continue

            # Plot subplots. 
            ax.scatter(floodPercents_real[idx], myFloodPercents[key][j], color='red', s=50)
            forRMSE.append(myFloodPercents[key][j] - floodPercents_real[idx])

            if floodPercents_real[idx] > max_fP:
                max_fP = floodPercents_real[idx]
            if myFloodPercents[key][j] > max_fP: 
                max_fP = myFloodPercents[key][j]

        # add labels and the title
        ax.set_ylabel('Forecasted Flood Percentage')
        ax.set_xlabel('Observed Flood Percentage')
        ax.set_title(key)

        # plot straight line to make clear what is an over/under forecast. 
        # max_fP = max(floodPercents_real)
        # if max(myFloodPercents[key]) > max_fP:
        #     max_fP = max(myFloodPercents[key])
        ax.plot([0, max_fP], [0, max_fP], color='black')
        ax.grid()
        ax.set_aspect('equal')
        # print(f'county = {county}')
    
        # calculate and display RMSE
        myRMSE = np.sqrt( np.mean( np.square(forRMSE) ) )
        ax.text(max_fP/4, max_fP*(3/4), f'RMSE = {myRMSE:.1f}')

        # adjust subplot spacing so it's all readable
        fig.subplots_adjust(hspace = 0.4, wspace = 0.4)
        
        i = i + 1
    
    return fig

In [ ]:
# construct the dictionary to pass to the plotter
myFP = {
    "Multi-mode Polynomial": fp_poly_multi, 
    "Single-mode Polynomial": fp_poly_single,
    "Multi-mode Neural Network": fp_nn_multi,
    "Single-mode Neural Network": fp_nn_single
}

# Run plotter and assign figure to object. 
rVoFP = realVSobservedFloodPercentages(da_poly_multi.time.values, time_flood, floodpercent, myFP)

***
## Save visualizations of forecast results

This code will save the plots of: 
* Visual comparisons of each flood forecast to their corresponding observed water extent. 
* Observed and forecasted flood percentages through time.
* RMSE of observed vs forecasted flood percentages. 

Additionally, it will save the statistics calculated for each forecast and flood observation. 

In [ ]:
### --- Save visualizations and statistics --- ###


# set up save location names
saveLoc_sM_nn   = Path(fc.selected, 'Figures',myLoc,f'sM_nn_smoothing-{smoothing_frame[myIdx]:02d}days')
saveLoc_mM_nn   = Path(fc.selected, 'Figures',myLoc,f'mM-#modes={np.max(myCombo_nn)}of{np.max(modes_nn)}_nn_smoothing-{smoothing_frame[myIdx]:02d}days')
saveLoc_sM_poly = Path(fc.selected, 'Figures',myLoc,f'sM_poly_smoothing-{smoothing_frame[myIdx]:02d}days')
saveLoc_mM_poly = Path(fc.selected, 'Figures',myLoc,f'mM-#modes={np.max(myCombo_poly)}of{np.max(modes_poly)}_poly_smoothing-{smoothing_frame[myIdx]:02d}days')

# creat save locations if they don't exist
saveLoc_sM_nn.mkdir(parents=True, exist_ok=True)
saveLoc_mM_nn.mkdir(parents=True, exist_ok=True)
saveLoc_sM_poly.mkdir(parents=True, exist_ok=True)
saveLoc_mM_poly.mkdir(parents=True, exist_ok=True)

# save the statistics for later inspection
stats2csv( Path(saveLoc_sM_nn,'statistics_singleMode_nn.csv'), statistics_header, statistics_singleMode_nn, forecasting_files)
stats2csv( Path(saveLoc_mM_nn,'statistics_multiMode_nn.csv'), statistics_header, statistics_multiMode_nn, forecasting_files)
stats2csv( Path(saveLoc_sM_poly,'statistics_singleMode_poly.csv'), statistics_header, statistics_singleMode_poly, forecasting_files)
stats2csv( Path(saveLoc_mM_poly,'statistics_multiMode_poly.csv'), statistics_header, statistics_multiMode_poly, forecasting_files)

# # save flood percentage figures
myFldPcnt_singleMode_nn.savefig(Path(saveLoc_sM_nn,'floodPercent_sM_nn.png'))
myFldPcnt_multiMode_nn.savefig(Path(saveLoc_mM_nn,'floodPercent_mM_nn.png'))
myFldPcnt_singleMode_poly.savefig(Path(saveLoc_sM_poly,'floodPercent_sM_poly.png'))
myFldPcnt_multiMode_poly.savefig(Path(saveLoc_mM_poly,'floodPercent_mM_poly.png'))

# save RMSE figure (put it into every folder)
rVoFP.savefig(Path(saveLoc_sM_nn,'RMSE.png'))
rVoFP.savefig(Path(saveLoc_mM_nn,'RMSE.png'))
rVoFP.savefig(Path(saveLoc_sM_poly,'RMSE.png'))
rVoFP.savefig(Path(saveLoc_mM_poly,'RMSE.png'))

# save the figures for later display
for i in range(0,len(myFigs_singleMode_nn)): 
    # set up file name for each type
    flnm_sM_nn   = f'smoothing-{smoothing_frame[myIdx]:02d}_date-{statistics_singleMode_nn[i][0]}.png'
    flnm_mM_nn   = f'smoothing-{smoothing_frame[myIdx]:02d}_date-{statistics_multiMode_nn[i][0]}.png'
    flnm_sM_poly = f'smoothing-{smoothing_frame[myIdx]:02d}_date-{statistics_multiMode_poly[i][0]}.png'
    flnm_mM_poly = f'smoothing-{smoothing_frame[myIdx]:02d}_date-{statistics_multiMode_poly[i][0]}.png'
    # save figures
    myFigs_singleMode_nn[i].savefig(Path(saveLoc_sM_nn,flnm_sM_nn), bbox_inches='tight')
    myFigs_multiMode_nn[i].savefig(Path(saveLoc_mM_nn,flnm_mM_nn), bbox_inches='tight')
    myFigs_singleMode_poly[i].savefig(Path(saveLoc_sM_poly,flnm_sM_poly), bbox_inches='tight')
    myFigs_multiMode_poly[i].savefig(Path(saveLoc_mM_poly,flnm_sM_poly), bbox_inches='tight')

In [ ]:
# Function to create the animated gifs
import matplotlib.animation as animation
def animate(dataset, fps, flnm, type_file, rewrite=False):
    
    if os.path.exists(flnm) and rewrite==False:
        print("GIF already exists.")
    else:
        # Create the animation
        fig = plt.figure()
        num_frames = dataset.shape[0]-1 # Because of Python's indexing and HTML indexing difference
        ani = animation.FuncAnimation(fig, update_plot, frames=num_frames, fargs=(dataset,type_file,), blit=False)

        # To show the animation in Jupyter Notebook (optional)

        from IPython.display import HTML
        HTML(ani.to_jshtml())

        # Save the animation as a GIF
        ani.save(flnm, writer='pillow', fps=fps)
        plt.close(ani._fig)
        plt.close(fig)

# Function that plots one image at a time
def update_plot(frame, dataset, type_file):
    # Extract the 2D slice corresponding to the current time step
    current_slice = dataset[frame, :, :]
    
    # Clear the previous plot
    plt.clf()
    
    # Depending on if your input iw water mask or RTC, the colormap changes
    if type_file == 'Water_Masks':
        plt.imshow(current_slice, cmap='Blues', interpolation="none", vmin=0, vmax=1.2)
    else:
        # Plot the 2D slice
        plt.imshow(current_slice, cmap='viridis', vmin = dataset.mean() - dataset.std(), vmax = dataset.mean() + dataset.std())
    
    # Add a title and other plot decorations (optional)
    plt.colorbar()
    
    # Add the date as a subtitle
    current_date = str(dataset.time[frame].values)[:10]  # Assuming time is in datetime format
    plt.text(0.5, 1.01, f'Date: {current_date}', transform=plt.gca().transAxes,
             ha='center', va='bottom', fontsize=10)

In [ ]:
# Production! 
# set up file save names
flnm_sM_poly_gif = Path(saveLoc_sM_poly,'sM-poly_gif-forecast_.gif')
flnm_mM_poly_gif = Path(saveLoc_mM_poly,'mM-poly_gif-forecast_.gif')
flnm_sM_nn_gif = Path(saveLoc_sM_nn,'sM-nn_gif-forecast_.gif')
flnm_mM_nn_gif = Path(saveLoc_mM_nn,'mM-nn_gif-forecast_.gif')

# set number of frames per second
fps = 2 # frames per second

# save the gifs
animate(da_poly_single[0:], fps, flnm_sM_poly_gif, 'Water_Masks', rewrite=True)
print('Single mode poly complete.')
animate(da_poly_multi[0:], fps, flnm_mM_poly_gif, 'Water_Masks', rewrite=True)
print('Multi mode poly complete.')
animate(da_nn_single[0:], fps, flnm_sM_nn_gif, 'Water_Masks', rewrite=True)
print('Single mode NN complete.')
animate(da_nn_multi[0:], fps, flnm_mM_nn_gif, 'Water_Masks', rewrite=True)
print('Multi mode NN complete.')